In [1]:
# This cell loads the evaluated TREC DL queries and creates word-count length groups for the subgroup analysis.
import pyterrier as pt
import pandas as pd

dataset = pt.get_dataset("msmarco_passage")
index = dataset.get_index(variant="terrier_stemmed")

# Load DL-hard annotations
annotations_url = "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/annotations/query/annotations.tsv"
annotations = pd.read_csv(
    annotations_url,
    sep="	",
    header=None,
    names=["qid", "query", "intent", "answer_type", "topic_domain", "serp_type"],
)
annotations["qid"] = annotations["qid"].astype(str)

# Load TREC DL 2019/2020 qrels
dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020")
all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
all_qrels["qid"] = all_qrels["qid"].astype(str)

# Keep only queries that have both annotations and relevance judgements
qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)

print(f"Queries with both annotations and relevance values: {len(topics)}")
print(topics["intent"].value_counts())
print("")
print(topics["answer_type"].value_counts())

# Word-count based length grouping (standard in IR research).
# Character-based bins are too granular for ~97 queries and produce
# many near-empty groups that break statistical tests.
# Three balanced word-count groups give ~25-40 queries each.
topics["query_word_count"] = topics["query"].str.split().str.len()

def length_group(n):
    if n <= 3:
        return "Short (<=3 words)"
    elif n <= 6:
        return "Medium (4-6 words)"
    else:
        return "Long (7+ words)"

topics["query_length_group"] = topics["query_word_count"].apply(length_group)
print("Query length group distribution:")
print(topics["query_length_group"].value_counts())
print("Word count stats:")
print(topics["query_word_count"].describe())


Queries with both annotations and relevance values: 97
intent
description     48
list            10
quantity         9
reason           8
attribute        7
entity           5
verification     3
process          3
language         3
weather          1
Name: count, dtype: int64

answer_type
factoid              28
definition           21
long answer          16
list                 10
short description     7
comparison            5
short answer          5
guide                 2
weather               1
Long answer           1
multi-answer          1
Name: count, dtype: int64
Query length group distribution:
query_length_group
Medium (4-6 words)    57
Long (7+ words)       27
Short (<=3 words)     13
Name: count, dtype: int64
Word count stats:
count    97.000000
mean      5.752577
std       2.231549
min       2.000000
25%       4.000000
50%       5.000000
75%       7.000000
max      15.000000
Name: query_word_count, dtype: float64


In [2]:
# This cell cleans the queries and initializes the BM25, RM3, and Bo1 retrieval pipelines.
import re

def clean_query(q):
    q = re.sub(r"[^\w\s]", " ", str(q))
    q = re.sub(r"\s+", " ", q).strip()
    return q

# Clean queries (else it gives errors in PyTerrier)
topics["query"] = topics["query"].apply(clean_query)

# Initialize BM25 retriever
bm25 = pt.terrier.Retriever(index, wmodel="BM25")

# Initialize classic query expansion models
rm3 = pt.rewrite.RM3(index)
bo1 = pt.rewrite.Bo1QueryExpansion(index)

bm25_rm3 = bm25 >> rm3 >> bm25
bm25_bo1 = bm25 >> bo1 >> bm25


Java started (triggered by Retriever.__init__) and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [3]:
# This cell runs BM25, traditional QE, and LLM QE on the shared query set,
# then builds Query2Doc-vs-baseline per-query delta tables for the length analysis.
from pyterrier.measures import RR, nDCG, AP, R

target_method = "Q2D_Llama3"
traditional_methods = ["BM25", "RM3", "Bo1"]
metric_order = ["nDCG@10", "AP(rel=2)", "RR(rel=2)@10", "R(rel=2)@100"]

results_classical = pt.Experiment(
    [bm25, bm25_rm3, bm25_bo1],
    topics[["qid", "query"]],
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["BM25", "RM3", "Bo1"],
    perquery=True,
)

q2d_path = "query2doc/topics_with_q2d.csv"
q2d_topics = pd.read_csv(q2d_path)
q2d_topics["qid"] = q2d_topics["qid"].astype(str)
q2d_topics = q2d_topics.rename(columns={"q2d_query": "query"})
q2d_topics["query"] = q2d_topics["query"].apply(clean_query)

results_q2d = pt.Experiment(
    [bm25],
    q2d_topics[["qid", "query"]],
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=[target_method],
    perquery=True,
)

cot_path = "cot/topics_with_cot.csv"
cot_topics = pd.read_csv(cot_path)
cot_topics["qid"] = cot_topics["qid"].astype(str)
cot_topics["cot_query"] = cot_topics["cot_query"].apply(clean_query)

results_cot = pt.Experiment(
    [bm25],
    cot_topics[["qid", "cot_query"]].rename(columns={"cot_query": "query"}),
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["CoT_Llama3"],
    perquery=True,
)

results = pd.concat([results_classical, results_q2d, results_cot], ignore_index=True)
label_map = topics[["qid", "query", "query_length_group", "query_word_count"]]
results = results.merge(label_map, on="qid", how="left")

def build_delta_frame(results_df, target_name, baseline_names, metadata_cols):
    delta_rows = []
    for measure in metric_order:
        measure_df = results_df[results_df["measure"] == measure]
        score_pivot = measure_df.pivot(index="qid", columns="name", values="value")
        meta_df = measure_df[["qid"] + metadata_cols].drop_duplicates("qid")
        merged = meta_df.merge(score_pivot.reset_index(), on="qid", how="left")
        if target_name not in merged.columns:
            continue

        for baseline_name in baseline_names:
            if baseline_name not in merged.columns:
                continue
            paired = merged[["qid"] + metadata_cols + [target_name, baseline_name]].dropna().copy()
            paired["comparison"] = f"{target_name} - {baseline_name}"
            paired["measure"] = measure
            paired["target_score"] = paired[target_name]
            paired["baseline_score"] = paired[baseline_name]
            paired["delta"] = paired["target_score"] - paired["baseline_score"]
            delta_rows.append(
                paired[
                    [
                        "qid",
                        *metadata_cols,
                        "comparison",
                        "measure",
                        "target_score",
                        "baseline_score",
                        "delta",
                    ]
                ]
            )

    if not delta_rows:
        return pd.DataFrame(
            columns=[
                "qid",
                *metadata_cols,
                "comparison",
                "measure",
                "target_score",
                "baseline_score",
                "delta",
            ]
        )

    return pd.concat(delta_rows, ignore_index=True)

length_delta_df = build_delta_frame(
    results,
    target_method,
    traditional_methods,
    ["query", "query_length_group", "query_word_count"],
)

print("=== Overall Results ===")
overall = results.groupby(["name", "measure"])["value"].mean().unstack()
print(overall)

print("=== nDCG@10 by Query Length ===")
ndcg_by_length = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "query_length_group"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_length)

print("=== Mean Query2Doc Gain by Query Length ===")
length_gain_summary = (
    length_delta_df[length_delta_df["measure"] == "nDCG@10"]
    .groupby(["comparison", "query_length_group"])["delta"]
    .mean()
    .unstack()
)
print(length_gain_summary)


=== Overall Results ===
measure     AP(rel=2)  R(rel=2)@100  RR(rel=2)@10   nDCG@10
name                                                       
BM25         0.290089      0.541421      0.625749  0.487382
Bo1          0.311664      0.568762      0.618295  0.500851
CoT_Llama3   0.338962      0.559121      0.643941  0.514828
Q2D_Llama3   0.389330      0.628159      0.756554  0.578139
RM3          0.317579      0.575153      0.628371  0.516954
=== nDCG@10 by Query Length ===
query_length_group  Long (7+ words)  Medium (4-6 words)  Short (<=3 words)
name                                                                      
BM25                       0.510299            0.467334           0.527691
Bo1                        0.530557            0.478969           0.535098
CoT_Llama3                 0.502199            0.498091           0.614446
Q2D_Llama3                 0.553513            0.572956           0.652015
RM3                        0.548141            0.489776           0.571343

In [4]:
# This cell tests whether Query2Doc gains differ by query length,
# and keeps within-group paired Q2D-vs-traditional comparisons as secondary evidence.
from scipy.stats import kruskal, mannwhitneyu, wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations

comparison_order = [f"{target_method} - {baseline}" for baseline in traditional_methods]
length_order = [
    group
    for group in ["Short (<=3 words)", "Medium (4-6 words)", "Long (7+ words)"]
    if group in length_delta_df["query_length_group"].dropna().unique()
]

def apply_holm(df, p_col="p_value", group_cols=None):
    df = df.copy()
    if len(df) == 0:
        df["p_corrected"] = pd.Series(dtype=float)
        df["significant"] = pd.Series(dtype=bool)
        return df

    if group_cols:
        corrected_parts = []
        for _, part in df.groupby(group_cols, dropna=False, sort=False):
            reject, p_corr, _, _ = multipletests(part[p_col], method="holm")
            part = part.copy()
            part["p_corrected"] = p_corr
            part["significant"] = reject
            corrected_parts.append(part)
        return pd.concat(corrected_parts, ignore_index=True)

    reject, p_corr, _, _ = multipletests(df[p_col], method="holm")
    df["p_corrected"] = p_corr
    df["significant"] = reject
    return df

def run_overall_paired_tests(results_df, target_name, baseline_names):
    rows = []
    for measure in metric_order:
        measure_df = results_df[results_df["measure"] == measure]
        score_pivot = measure_df.pivot(index="qid", columns="name", values="value")
        if target_name not in score_pivot.columns:
            continue

        for baseline_name in baseline_names:
            if baseline_name not in score_pivot.columns:
                continue
            paired = score_pivot[[target_name, baseline_name]].dropna()
            if len(paired) < 2:
                continue

            diffs = paired[target_name] - paired[baseline_name]
            if (diffs == 0).all():
                stat, p = 0.0, 1.0
            else:
                stat, p = wilcoxon(paired[target_name], paired[baseline_name], alternative="two-sided")

            rows.append(
                {
                    "comparison": f"{target_name} - {baseline_name}",
                    "measure": measure,
                    "n_pairs": len(paired),
                    "target_mean": round(float(paired[target_name].mean()), 4),
                    "baseline_mean": round(float(paired[baseline_name].mean()), 4),
                    "mean_delta": round(float(diffs.mean()), 4),
                    "median_delta": round(float(diffs.median()), 4),
                    "wilcoxon_stat": round(float(stat), 4),
                    "p_value": float(p),
                }
            )

    overall_df = apply_holm(pd.DataFrame(rows))
    if len(overall_df) > 0:
        overall_df["comparison"] = pd.Categorical(overall_df["comparison"], categories=comparison_order, ordered=True)
        overall_df["measure"] = pd.Categorical(overall_df["measure"], categories=metric_order, ordered=True)
    return overall_df

def run_delta_group_tests(delta_df, group_col, group_order):
    omnibus_rows = []
    pairwise_rows = []

    for comparison in comparison_order:
        for measure in metric_order:
            subset = delta_df[
                (delta_df["comparison"] == comparison)
                & (delta_df["measure"] == measure)
                & (delta_df[group_col].notna())
            ]

            group_scores = {}
            for group_name in group_order:
                scores = subset.loc[subset[group_col] == group_name, "delta"].dropna().values
                if len(scores) >= 2:
                    group_scores[group_name] = scores

            if len(group_scores) < 2:
                continue

            kw_stat, kw_p = kruskal(*[group_scores[group_name] for group_name in group_scores])
            omnibus_rows.append(
                {
                    "comparison": comparison,
                    "measure": measure,
                    "kw_statistic": round(float(kw_stat), 4),
                    "p_value": float(kw_p),
                    "group_means": {
                        group_name: round(float(group_scores[group_name].mean()), 4)
                        for group_name in group_scores
                    },
                }
            )

            for group_1, group_2 in combinations(group_scores.keys(), 2):
                scores_1 = group_scores[group_1]
                scores_2 = group_scores[group_2]
                stat, p = mannwhitneyu(scores_1, scores_2, alternative="two-sided")
                pairwise_rows.append(
                    {
                        "comparison": comparison,
                        "measure": measure,
                        "group_1": group_1,
                        "group_2": group_2,
                        "n_group_1": len(scores_1),
                        "n_group_2": len(scores_2),
                        "mean_group_1": round(float(scores_1.mean()), 4),
                        "mean_group_2": round(float(scores_2.mean()), 4),
                        "delta_gap": round(float(scores_1.mean() - scores_2.mean()), 4),
                        "mw_statistic": round(float(stat), 4),
                        "p_value": float(p),
                    }
                )

    omnibus_df = apply_holm(pd.DataFrame(omnibus_rows))
    pairwise_df = apply_holm(pd.DataFrame(pairwise_rows), group_cols=["comparison", "measure"])

    if len(omnibus_df) > 0:
        omnibus_df["comparison"] = pd.Categorical(omnibus_df["comparison"], categories=comparison_order, ordered=True)
        omnibus_df["measure"] = pd.Categorical(omnibus_df["measure"], categories=metric_order, ordered=True)
    if len(pairwise_df) > 0:
        pairwise_df["comparison"] = pd.Categorical(pairwise_df["comparison"], categories=comparison_order, ordered=True)
        pairwise_df["measure"] = pd.Categorical(pairwise_df["measure"], categories=metric_order, ordered=True)
    return omnibus_df, pairwise_df

def run_within_group_paired_tests(results_df, group_col, group_order, target_name, baseline_names):
    rows = []
    for group_name in group_order:
        group_df = results_df[results_df[group_col] == group_name]
        for measure in metric_order:
            score_pivot = (
                group_df[group_df["measure"] == measure]
                .pivot(index="qid", columns="name", values="value")
            )
            if target_name not in score_pivot.columns:
                continue

            for baseline_name in baseline_names:
                if baseline_name not in score_pivot.columns:
                    continue
                paired = score_pivot[[target_name, baseline_name]].dropna()
                if len(paired) < 2:
                    continue

                diffs = paired[target_name] - paired[baseline_name]
                if (diffs == 0).all():
                    stat, p = 0.0, 1.0
                else:
                    stat, p = wilcoxon(paired[target_name], paired[baseline_name], alternative="two-sided")

                rows.append(
                    {
                        "group": group_name,
                        "comparison": f"{target_name} - {baseline_name}",
                        "measure": measure,
                        "n_pairs": len(paired),
                        "target_mean": round(float(paired[target_name].mean()), 4),
                        "baseline_mean": round(float(paired[baseline_name].mean()), 4),
                        "mean_delta": round(float(diffs.mean()), 4),
                        "median_delta": round(float(diffs.median()), 4),
                        "wilcoxon_stat": round(float(stat), 4),
                        "p_value": float(p),
                    }
                )

    within_df = apply_holm(pd.DataFrame(rows), group_cols=["group"])
    if len(within_df) > 0:
        within_df["group"] = pd.Categorical(within_df["group"], categories=group_order, ordered=True)
        within_df["comparison"] = pd.Categorical(within_df["comparison"], categories=comparison_order, ordered=True)
        within_df["measure"] = pd.Categorical(within_df["measure"], categories=metric_order, ordered=True)
    return within_df

overall_delta_tests = run_overall_paired_tests(results, target_method, traditional_methods)
print("=== Overall paired Query2Doc vs baseline comparisons (Holm-corrected) ===")
print(overall_delta_tests.sort_values(["measure", "p_corrected"]).to_string(index=False))

length_omnibus, length_pairwise = run_delta_group_tests(length_delta_df, "query_length_group", length_order)
print("=== Query2Doc gain differences by Query Length (Holm-corrected omnibus) ===")
print(length_omnibus.sort_values(["measure", "p_corrected"]).to_string(index=False))

print("=== Significant pairwise length differences in Query2Doc gains ===")
sig_length_pairwise = length_pairwise[length_pairwise["significant"]].sort_values(["measure", "comparison", "p_corrected"])
if len(sig_length_pairwise) == 0:
    print("No significant pairwise length differences after Holm correction.")
else:
    print(
        sig_length_pairwise[
            [
                "comparison",
                "measure",
                "group_1",
                "group_2",
                "n_group_1",
                "n_group_2",
                "mean_group_1",
                "mean_group_2",
                "delta_gap",
                "p_corrected",
            ]
        ].to_string(index=False)
    )

length_within_tests = run_within_group_paired_tests(
    results,
    "query_length_group",
    length_order,
    target_method,
    traditional_methods,
)
print("=== Secondary within-length paired Query2Doc vs baseline tests ===")
print(length_within_tests.sort_values(["group", "measure", "p_corrected"]).to_string(index=False))


=== Overall paired Query2Doc vs baseline comparisons (Holm-corrected) ===
       comparison      measure  n_pairs  target_mean  baseline_mean  mean_delta  median_delta  wilcoxon_stat      p_value  p_corrected  significant
Q2D_Llama3 - BM25      nDCG@10       97       0.5781         0.4874      0.0908        0.0241          609.0 7.613743e-05 6.090994e-04         True
 Q2D_Llama3 - RM3      nDCG@10       97       0.5781         0.5170      0.0612        0.0289         1352.0 5.134446e-03 6.934447e-03         True
 Q2D_Llama3 - Bo1      nDCG@10       97       0.5781         0.5009      0.0773        0.0361         1366.0 1.689531e-03 6.934447e-03         True
Q2D_Llama3 - BM25    AP(rel=2)       97       0.3893         0.2901      0.0992        0.0344          397.0 3.408219e-08 4.089862e-07         True
 Q2D_Llama3 - Bo1    AP(rel=2)       97       0.3893         0.3117      0.0777        0.0372         1105.0 7.850457e-06 7.850457e-05         True
 Q2D_Llama3 - RM3    AP(rel=2)       9

In [5]:
# This cell is reserved for optional follow-up plots or failure-case inspection.
